<a href="https://colab.research.google.com/github/gopalpatil15/data-engineering-practice/blob/main/05_api_ingestion_currency_etl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests

url = "https://open.er-api.com/v6/latest/USD"

# 1. Make GET request to this URL:
response = requests.get(url)

# 2. Print status code
print(response.status_code)

# 3. Print data.keys()
data = response.json()
print(data.keys())

# 4. Print data['base_code']
print(data["base_code"])

# 5. Print how many currencies in data['rates']
print(len(data["rates"]))

200
dict_keys(['result', 'provider', 'documentation', 'terms_of_use', 'time_last_update_unix', 'time_last_update_utc', 'time_next_update_unix', 'time_next_update_utc', 'time_eol_unix', 'base_code', 'rates'])
USD
166


Task:
1. Access data['rates'] — it's a dictionary
   {currency_code: rate, ...}
   Example: {"INR": 83.5, "EUR": 0.92, ...}

2. Convert to DataFrame with 2 columns:
   currency_code | rate

3. Filter to keep only these 10 currencies:
   INR, EUR, GBP, JPY, AUD, CAD, SGD, AED, CHF, CNY

4. Reset index

5. Print the result

In [ ]:
import pandas as pd

# 1. Access the dictionary (assuming 'data' is your main dictionary)
rates_dict = data['rates']

# 2. Convert to DataFrame with 2 columns
df = pd.DataFrame(list(rates_dict.items()), columns=['currency_code', 'rate'])

# 3. Filter to keep only the 10 specific currencies
currencies_to_keep = ['INR', 'EUR', 'GBP', 'JPY', 'AUD', 'CAD', 'SGD', 'AED', 'CHF', 'CNY']
df = df[df['currency_code'].isin(currencies_to_keep)]

# 4. Reset the index (drop=True prevents the old index from becoming a new column)
df = df.reset_index(drop=True)

# 5. Print the result
print(df)

  currency_code        rate
0           AED    3.672500
1           AUD    1.439528
2           CAD    1.421183
3           CHF    0.805556
4           CNY    6.801784
5           EUR    0.874637
6           GBP    0.747968
7           INR   95.455992
8           JPY  162.128909
9           SGD    1.292448


Add to your DataFrame:
1. base_currency column → "USD" for all rows
2. rate_vs_inr → INR rate divided by each currency rate
   (how many INR = 1 unit of that currency)
3. strong_currency → True if rate_vs_inr > 50
4. Round rate to 4 decimal places
5. Print final DataFrame

In [ ]:
inr_rate = df.loc[df['currency_code'] == 'INR', 'rate'].values[0]
print(f"INR rate: {inr_rate}")  # should print ~95.45

df['base_currency'] = 'USD'
df['rate_vs_inr'] = inr_rate / df['rate']
df['strong_currency'] = df['rate_vs_inr'] > 50
df['rate'] = df['rate'].round(4)

print(df)

INR rate: 95.456
  currency_code      rate base_currency  rate_vs_inr  strong_currency
0           AED    3.6725           USD    25.992103            False
1           AUD    1.4395           USD    66.311914             True
2           CAD    1.4212           USD    67.165775             True
3           CHF    0.8056           USD   118.490566             True
4           CNY    6.8018           USD    14.033932            False
5           EUR    0.8746           USD   109.142465             True
6           GBP    0.7480           USD   127.614973             True
7           INR   95.4560           USD     1.000000            False
8           JPY  162.1289           USD     0.588766            False
9           SGD    1.2924           USD    73.859486             True


1. Create SQLite engine
2. Load df to table "exchange_rates"
3. Verify with SELECT COUNT(*)

4. SQL Query 1:
   Show all strong currencies ordered by rate_vs_inr DESC

5. SQL Query 2:
   Which currency gives most INR per unit?
   (strongest against INR)

6. SQL Query 3:
   RANK all currencies by rate_vs_inr using RANK()

In [ ]:
from sqlalchemy import create_engine, text

#-------------task 1 Initializing the Database ------------------
engine = create_engine('sqlite:///liveapi.db')

#------------Task 1 — Load to SQLite -------------
try:
  df.to_sql('exchange_rates', con=engine, if_exists='replace', index=False)

except NameError:
  print("Error: df is not defined. Cannot create 'exchange_rates' table.")

with engine.connect() as conn:
  result = conn.execute(text("SELECT COUNT(*) FROM exchange_rates"))
  row_count = result.fetchone()[0]
  print(f"Number of rows in 'exchange_rates' table: {row_count}")

query1 = """
SELECT *
FROM exchange_rates
WHERE strong_currency = 1
ORDER BY rate_vs_inr DESC
"""

query2 = """
SELECT currency_code, rate_vs_inr
FROM exchange_rates
WHERE strong_currency = 1
ORDER BY rate_vs_inr DESC
LIMIT 1
"""

query3 = """
SELECT currency_code, rate_vs_inr,
       RANK() OVER (ORDER BY rate_vs_inr DESC) AS rank
FROM exchange_rates
WHERE strong_currency = 1
"""


print("Query 1:")
with engine.connect() as conn:
  result = conn.execute(text(query1))
  for row in result:
    print(row)

print("\nQuery 2:")
with engine.connect() as conn:
  result = conn.execute(text(query2))
  for row in result:
    print(row)


print("\nQuery 3:")
with engine.connect() as conn:
  result = conn.execute(text(query3))
  for row in result:
    print(row)


Number of rows in 'exchange_rates' table: 10
Query 1:
('GBP', 0.748, 'USD', 127.6149732620321, 1)
('CHF', 0.8056, 'USD', 118.49056603773586, 1)
('EUR', 0.8746, 'USD', 109.14246512691516, 1)
('SGD', 1.2924, 'USD', 73.85948622717426, 1)
('CAD', 1.4212, 'USD', 67.16577540106952, 1)
('AUD', 1.4395, 'USD', 66.31191385897881, 1)

Query 2:
('GBP', 127.6149732620321)

Query 3:
('GBP', 127.6149732620321, 1)
('CHF', 118.49056603773586, 2)
('EUR', 109.14246512691516, 3)
('SGD', 73.85948622717426, 4)
('CAD', 67.16577540106952, 5)
('AUD', 66.31191385897881, 6)
